In [1]:
import os
print("CWD:", os.getcwd())

CWD: C:\Users\vaio hd


In [2]:
!pip install flask pandas openpyxl

In [3]:
import os
from pathlib import Path

project_dir = Path("task_crud_excel_json")
project_dir.mkdir(exist_ok=True)

(project_dir / "templates").mkdir(exist_ok=True)

print("Created:", project_dir.resolve())
print("Templates:", (project_dir/"templates").resolve())

Created: C:\Users\vaio hd\task_crud_excel_json
Templates: C:\Users\vaio hd\task_crud_excel_json\templates


In [4]:
base_html = """<!doctype html>
<html lang="fa" dir="rtl">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Task Manager</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.rtl.min.css" rel="stylesheet">
</head>
<body class="bg-light">
  <nav class="navbar navbar-expand-lg navbar-dark bg-dark">
    <div class="container">
      <a class="navbar-brand" href="/">Task Manager</a>
    </div>
  </nav>

  <main class="container py-4">
    {% block content %}{% endblock %}
  </main>

  <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
"""
(project_dir/"templates"/"base.html").write_text(base_html, encoding="utf-8")
print("Wrote base.html")

Wrote base.html


In [5]:
index_html = """{% extends "base.html" %}
{% block content %}

<div class="row g-4">
  <div class="col-lg-5">
    <div class="card shadow-sm">
      <div class="card-header bg-primary text-white">
        <h5 class="mb-0">Add Task</h5>
      </div>

      <div class="card-body">
        <form method="POST" action="/tasks">
          <div class="mb-3">
            <label class="form-label">عنوان وظیفه</label>
            <input type="text" name="title" class="form-control" required>
          </div>

          <div class="mb-3">
            <label class="form-label">توضیحات</label>
            <input type="text" name="description" class="form-control">
          </div>

          <div class="mb-3">
            <label class="form-label">وضعیت</label>
            <select name="status" class="form-select">
              <option value="Pending" selected>Pending</option>
              <option value="Completed">Completed</option>
            </select>
          </div>

          <button type="submit" class="btn btn-success w-100">Add (Create)</button>
        </form>

        <hr>

        <h6 class="mb-3">Import Excel</h6>
        <form method="POST" action="/import_excel" enctype="multipart/form-data">
          <div class="mb-3">
            <label class="form-label">فایل Excel (.xlsx)</label>
            <input type="file" name="excel_file" class="form-control" accept=".xlsx" required>
          </div>
          <button type="submit" class="btn btn-primary w-100">Import Excel</button>
        </form>

        <div class="mt-3">
          <a class="btn btn-outline-dark w-100" href="/download_json">Download as JSON</a>
        </div>

        <small class="text-muted d-block mt-3">
          ستون‌های موردنیاز Excel: <b>Title</b>, <b>Description</b>, <b>Status</b>
        </small>
      </div>
    </div>
  </div>

  <div class="col-lg-7">
    <div class="card shadow-sm">
      <div class="card-header bg-dark text-white d-flex align-items-center justify-content-between">
        <h5 class="mb-0">Task List</h5>
        <span class="badge bg-light text-dark">{{ tasks|length }} items</span>
      </div>

      <div class="card-body">
        {% if tasks %}
          <div class="table-responsive">
            <table class="table table-striped align-middle">
              <thead class="table-light">
                <tr>
                  <th>ID</th>
                  <th>عنوان</th>
                  <th>توضیحات</th>
                  <th>وضعیت</th>
                  <th style="width: 260px;">Actions</th>
                </tr>
              </thead>
              <tbody>
                {% for t in tasks %}
                  <tr>
                    <td>{{ t.id }}</td>
                    <td>{{ t.title }}</td>
                    <td style="max-width: 320px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;">
                      {{ t.description }}
                    </td>
                    <td>
                      {% if t.status == "Completed" %}
                        <span class="badge text-bg-success">Completed</span>
                      {% else %}
                        <span class="badge text-bg-warning">Pending</span>
                      {% endif %}
                    </td>
                    <td>
                      <div class="d-flex flex-wrap gap-2">
                        <a class="btn btn-sm btn-outline-primary" href="/tasks/{{t.id}}">Read</a>
                        <a class="btn btn-sm btn-outline-warning" href="/tasks/{{t.id}}/edit">Update</a>
                        <form method="POST" action="/tasks/{{t.id}}/delete" onsubmit="return confirm('Delete this task?')">
                          <button class="btn btn-sm btn-outline-danger" type="submit">Delete</button>
                        </form>
                      </div>
                    </td>
                  </tr>
                {% endfor %}
              </tbody>
            </table>
          </div>
        {% else %}
          <p class="text-muted mb-0">هنوز رکوردی وجود ندارد. از فرم بالا یا Import Excel استفاده کنید.</p>
        {% endif %}
      </div>
    </div>

    {% if read_task %}
      <div class="card shadow-sm mt-4">
        <div class="card-header bg-secondary text-white">
          <h5 class="mb-0">Read</h5>
        </div>
        <div class="card-body">
          <p class="mb-1"><b>ID:</b> {{ read_task.id }}</p>
          <p class="mb-1"><b>Title:</b> {{ read_task.title }}</p>
          <p class="mb-1"><b>Description:</b> {{ read_task.description }}</p>
          <p class="mb-0"><b>Status:</b> {{ read_task.status }}</p>
        </div>
      </div>
    {% endif %}

  </div>
</div>

{% endblock %}
"""
(project_dir/"templates"/"index.html").write_text(index_html, encoding="utf-8")
print("Wrote index.html")

Wrote index.html


In [6]:
edit_html = """{% extends "base.html" %}
{% block content %}

<div class="card shadow-sm">
  <div class="card-header bg-warning text-dark">
    <h5 class="mb-0">Update Task (ID: {{ task.id }})</h5>
  </div>

  <div class="card-body">
    <form method="POST" action="">
      <div class="mb-3">
        <label class="form-label">عنوان وظیفه</label>
        <input type="text" name="title" class="form-control" value="{{ task.title }}" required>
      </div>

      <div class="mb-3">
        <label class="form-label">توضیحات</label>
        <input type="text" name="description" class="form-control" value="{{ task.description }}">
      </div>

      <div class="mb-3">
        <label class="form-label">وضعیت</label>
        <select name="status" class="form-select">
          <option value="Pending" {% if task.status == "Pending" %}selected{% endif %}>Pending</option>
          <option value="Completed" {% if task.status == "Completed" %}selected{% endif %}>Completed</option>
        </select>
      </div>

      <div class="d-flex gap-2">
        <button class="btn btn-success" type="submit">Save</button>
        <a class="btn btn-outline-secondary" href="/">Cancel</a>
      </div>
    </form>
  </div>
</div>

{% endblock %}
"""
(project_dir/"templates"/"edit_task.html").write_text(edit_html, encoding="utf-8")
print("Wrote edit_task.html")

Wrote edit_task.html


In [7]:
app_py = """import json
from io import BytesIO

import pandas as pd
from flask import Flask, render_template, request, redirect, url_for, send_file, abort

app = Flask(__name__)
app.secret_key = "change-me"

# ذخیره‌سازی داخل حافظه
tasks = []
next_id = 1

def normalize_status(s: str) -> str:
    if s is None:
        return "Pending"
    s = str(s).strip().lower()
    if s in ["completed", "done", "true", "1"]:
        return "Completed"
    return "Pending"

@app.route("/", methods=["GET"])
def index():
    return render_template("index.html", tasks=tasks)

@app.route("/tasks", methods=["POST"])
def create_task():
    global next_id

    title = (request.form.get("title") or "").strip()
    description = (request.form.get("description") or "").strip()
    status = normalize_status(request.form.get("status"))

    if not title:
        return redirect(url_for("index"))

    tasks.append({
        "id": next_id,
        "title": title,
        "description": description,
        "status": status
    })
    next_id += 1
    return redirect(url_for("index"))

@app.route("/tasks/<int:task_id>", methods=["GET"])
def read_task(task_id: int):
    task = next((t for t in tasks if t["id"] == task_id), None)
    if not task:
        abort(404)
    return render_template("index.html", tasks=tasks, read_task=task)

@app.route("/tasks/<int:task_id>/edit", methods=["GET", "POST"])
def edit_task(task_id: int):
    task = next((t for t in tasks if t["id"] == task_id), None)
    if not task:
        abort(404)

    if request.method == "POST":
        title = (request.form.get("title") or "").strip()
        description = (request.form.get("description") or "").strip()
        status = normalize_status(request.form.get("status"))

        if title:
            task["title"] = title
        task["description"] = description
        task["status"] = status
        return redirect(url_for("index"))

    return render_template("edit_task.html", task=task)

@app.route("/tasks/<int:task_id>/delete", methods=["POST"])
def delete_task(task_id: int):
    global tasks
    tasks = [t for t in tasks if t["id"] != task_id]
    return redirect(url_for("index"))

@app.route("/import_excel", methods=["POST"])
def import_excel():
    global next_id, tasks

    if "excel_file" not in request.files:
        return redirect(url_for("index"))

    file = request.files["excel_file"]
    if not file or file.filename == "":
        return redirect(url_for("index"))

    if not file.filename.lower().endswith(".xlsx"):
        return "Invalid file type. Please upload .xlsx", 400

    try:
        df = pd.read_excel(file)

        required = {"Title", "Description", "Status"}
        if not required.issubset(set(df.columns)):
            return "Excel must contain columns: Title, Description, Status", 400

        for _, row in df.iterrows():
            title = str(row.get("Title", "")).strip()
            if not title:
                continue

            description = str(row.get("Description", "")).strip()
            status = normalize_status(row.get("Status"))

            tasks.append({
                "id": next_id,
                "title": title,
                "description": description,
                "status": status
            })
            next_id += 1

        return redirect(url_for("index"))

    except Exception as e:
        return f"Error importing Excel: {e}", 500

@app.route("/download_json", methods=["GET"])
def download_json():
    json_text = json.dumps(tasks, ensure_ascii=False, indent=2)
    bio = BytesIO(json_text.encode("utf-8"))
    bio.seek(0)

    return send_file(
        bio,
        as_attachment=True,
        download_name="tasks.json",
        mimetype="application/json"
    )

if __name__ == "__main__":
    app.run(debug=True, port=5000)
"""
(project_dir/"app.py").write_text(app_py, encoding="utf-8")
print("Wrote app.py")

Wrote app.py


In [8]:
import os, sys, threading
from pathlib import Path

proj_dir = Path("task_crud_excel_json").resolve()
os.chdir(proj_dir)
sys.path.insert(0, str(proj_dir))

try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

from app import app

threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False),
    daemon=True
).start()

print("Open: http://127.0.0.1:5000/")

Open: http://127.0.0.1:5000/
 * Serving Flask app "app" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [02/Aug/2026 11:21:10] "GET / HTTP/1.1" 200 -


In [9]:
import pandas as pd

data = [
    {"Title": "خرید نان", "Description": "برای صبحانه", "Status": "Pending"},
    {"Title": "گزارش ماهانه", "Description": "ارسال به مدیر", "Status": "Completed"},
    {"Title": "تمرین ورزش", "Description": "۳۰ دقیقه دویدن", "Status": "Pending"},
]

df = pd.DataFrame(data)

out_path = "task_list_sample.xlsx"
df.to_excel(out_path, index=False)

print("Saved:", out_path)
df

Saved: task_list_sample.xlsx


,Title,Description,Status
0,خرید نان,برای صبحانه,Pending
1,گزارش ماهانه,ارسال به مدیر,Completed
2,تمرین ورزش,۳۰ دقیقه دویدن,Pending
